## Lab 6: Constituency Parsing with a CFG

#### The idea in one line

> One sentence can have **more than one parse tree**. We find the trees, convert them to the form CKY needs, and compare them by **spans**.

#### Our sentence

```
Book the flight through Houston
```

#### What we will do

| Step | What happens |
| --- | --- |
| 1. Parse | Write part of grammar L₁ and count the trees |
| 2. CNF and back | Binarize a tree, then restore it |
| 3. Spans | List each constituent's span [i, j] and compare the two meanings |

In [3]:
import nltk
from nltk import CFG

In [21]:
def get_grammar():
    grammar = CFG.fromstring("""
    S -> VP
    VP -> Verb NP
    VP -> VP PP
    NP -> Det Nominal
    NP -> ProperNoun
    Nominal -> Noun
    Nominal -> Nominal PP
    PP -> Preposition NP
    VP -> Verb NP PP

    Verb -> 'book' | 'Book'
    Det -> 'the'
    Noun -> 'flight'
    ProperNoun -> 'Houston'|'houston'
    Preposition -> 'through'
""")
    return grammar

In [22]:
words = "Book the flight through Houston".lower().split()

def get_parser(words="ArithmeticError"):  
    grammar = get_grammar()
    parser = nltk.ChartParser(grammar)
    trees = list(parser.parse(words))
    return trees

trees = get_parser(words)
for tree in trees:
    #print(tree)
    tree.pretty_print()
    

                S                              
                |                               
                VP                             
       _________|_________________              
      VP                          |            
  ____|___                        |             
 |        NP                      PP           
 |     ___|_____           _______|______       
 |    |      Nominal      |              NP    
 |    |         |         |              |      
Verb Det       Noun  Preposition     ProperNoun
 |    |         |         |              |      
book the      flight   through        houston  

                S                              
                |                               
                VP                             
  ______________|_________________              
 |        NP                      PP           
 |     ___|_____           _______|______       
 |    |      Nominal      |              NP    
 |    |         |         |   

In [38]:
tree=trees[1].copy(deep=True)
tree.chomsky_normal_form()
tree.pretty_print()
tree.un_chomsky_normal_form()
tree.pretty_print()
tree==trees[1]


                S                                         
                |                                          
                VP                                        
  ______________|________                                  
 |                   VP|<NP-PP>                           
 |         ______________|___________________              
 |        NP                                 PP           
 |     ___|_____                      _______|______       
 |    |      Nominal                 |              NP    
 |    |         |                    |              |      
Verb Det       Noun             Preposition     ProperNoun
 |    |         |                    |              |      
book the      flight              through        houston  

                S                              
                |                               
                VP                             
  ______________|_________________              
 |        NP                    

True

In [51]:
tree=trees[0]
for node in tree.subtrees():
    node_words = node.leaves()
    start_index = words.index(node_words[0])
    end_index = start_index + len(node_words)
    print(node.label(), start_index, end_index)

print("===================Other tree has same structure ==================")
tree=trees[2]
for node in tree.subtrees():
    node_words = node.leaves()
    start_index = words.index(node_words[0])
    end_index = start_index + len(node_words)
    print(node.label(), start_index, end_index)


S 0 5
VP 0 5
VP 0 3
Verb 0 1
NP 1 3
Det 1 2
Nominal 2 3
Noun 2 3
PP 3 5
Preposition 3 4
NP 4 5
ProperNoun 4 5
===================Other tree has same structure ==================
S 0 5
VP 0 5
Verb 0 1
NP 1 5
Det 1 2
Nominal 2 5
Nominal 2 3
Noun 2 3
PP 3 5
Preposition 3 4
NP 4 5
ProperNoun 4 5


## What we learned

- The sentence has **2 meanings**: the PP attaches to the verb or to the noun.
- CKY needs **binary** trees, and the original tree can be recovered.
- The two trees share **10 of 12** spans and differ in only 2.

#### Next lab: CKY

> CKY builds each span **once** in a table, finds **all** trees, and a **neural** score chooses the best one (Ch. 18, 18.6–18.7).